In [ ]:
import csv
import requests
from io import StringIO
from google.colab import files

additionalhint_append = False

SHEET_ID = "118Znzcf8Nr2y0-YPCnO3JbZX-QZhm39q7ATDTpouVjE"
GID = "2094187478"

SHEET_URL = (
    f"https://docs.google.com/spreadsheets/d/"
    f"{SHEET_ID}/export?format=csv&gid={GID}"
)

response = requests.get(SHEET_URL)
response.raise_for_status()

reader = list(csv.DictReader(StringIO(response.text), delimiter=","))


In [ ]:
ID = "Id"
ORIGEM_COL = "#Origem"

ORIGEM_TOG_PATCHED = "TOG patched"
ORIGEM_VANILLA = "Vanilla"
ORIGEM_NON_PATCHED = "Non-patched"
ORIGEM_LEGACY = "Legacy"

# Colunas que nunca entram no loop generico de chaves:
# - Id: usado como prefixo (Id.Chave = valor), nao e um campo em si
# - Description / TID_Description: par especial (vira T(TID, [[texto]])),
#   e so se aplica a arma TOG (ver get_weapon_lua_snippet)
ALWAYS_EXCLUDED = {"Id", "Description", "TID_Description"}

# Chaves dinamicas: tudo do cabecalho da planilha, exceto o que comeca com "#"
# e exceto as sempre-excluidas acima. Pra tirar uma coluna da geracao sem
# mexer neste script, basta prefixar o nome dela com "#" na planilha.
header = reader[0].keys() if reader else []
KEYS = [k for k in header if k not in ALWAYS_EXCLUDED and not k.startswith("#")]


def format_value(key, raw_value):
    """Formata o valor de uma celula pro formato Lua: aspas em string,
    listas {a, b, ...}, TRUE/FALSE -> true/false, numeros sem aspas."""
    value = raw_value
    if key == "Caliber":
        value = f'"{value}"'
    elif key == "AdditionalHint":
        # Valor ja vem pronto da planilha como expressao Lua (T(id, [[texto]])),
        # nao pode ganhar aspas ou vira string em vez de chamada de funcao.
        pass
    elif key == "PBbonus_base" or key == "Handling":
        # Pode ser negativo (ex: -4), e "-4".isdigit() e False, entao sem
        # essa excecao a regra generica de aspas quebraria o valor.
        pass
    elif key == "DisplayName":
        value = f'Untranslated([[{value}]])'
    elif value.startswith("{") and value.endswith("}"):
        elements = value[1:-1].split(", ")
        for i, el in enumerate(elements):
            if not el.replace(".", "", 1).isdigit():
                elements[i] = f'"{el}"'
        value = "{" + ", ".join(elements) + "}"
    elif value == "TRUE":
        value = "true"
    elif value == "FALSE":
        value = "false"
    elif not value.replace(".", "", 1).isdigit():
        value = f'"{value}"'
    return value


def handle_CTHModeProperties(prop, value, item_id):
    prefix_modes = {
        "oldcth_": "oldCTH",
        "acth_": "aCTH",
    }

    for prefix, mode in prefix_modes.items():
        if prop.startswith(prefix):
            actual_prop = prop[len(prefix):]
            modified_value = format_value(actual_prop, value)

            return (
                f'\tstoreProps({item_id}, "{actual_prop}", '
                f'{modified_value}, "{mode}", GBO_gCTHModeItemPropertyTable)'
            )

    return None


def get_weapon_lua_snippet(w, is_tog, is_legacy=False):
    """is_tog liga as duas logicas especiais do TOG:
    - rat_weigth tambem vira Weigth
    - Description/TID_Description vira T(TID, [[texto]])
    Nao ha mais filtro por is_tog_patched aqui dentro: quem decide se a
    arma e TOG ou GBO e a coluna #Origem, antes de chamar esta funcao."""
    result = ""

    for key in KEYS:
        if key == "ComponentSlots":
            result += f'\t{w[ID]}.{key} = {w[key]}\n'
            continue

        if key in w and w[key]:

            cth_result = handle_CTHModeProperties(key, w[key], w[ID])

            if cth_result:
                result += cth_result + "\n"
                continue

            modified_value = format_value(key, w[key])
            result += f'\t{w[ID]}.{key} = {modified_value}\n'

            #if key == "rat_weigth" and is_tog:
            #    result += f'\t{w[ID]}.Weigth = {modified_value}\n'

    #if is_tog and w.get("TID_Description"):
    if w.get("TID_Description"):
        result += f'\t{w[ID]}.Description = T({w["TID_Description"]}, [[{w["Description"]}]])\n'

    result += f'\t{w[ID]}.PenetrationClass = 1\n'

    # Legacy continua sendo tratado como TOG, mas nunca pode aparecer na loja.
    if is_legacy:
        result += f'\t{w[ID]}.CanAppearInShop = false\n'

    if additionalhint_append:
        result += f'\tif not string.find(["{w[ID]}"].AdditionalHint and ["{w[ID]}"].AdditionalHint[2] or "", "<description_hints>") then\n'
        result += f'\t\t["{w[ID]}"].AdditionalHint = T({{ "<description_hints>\\n" .. (["{w[ID]}"].AdditionalHint and ["{w[ID]}"].AdditionalHint[2] or "") }})\n'
        result += "\tend\n"

    return result

In [ ]:
LUA_FILE_NAME = "_PATCH_GBO_weapons.lua"


def write_lua_header(lua_file):
    """Cabecalho + storeProps, escritos uma unica vez no arquivo."""
    lua_file.write("-- ========== THIS IS AN AUTOMATICALLY GENERATED FILE! ==========\n\n")
    lua_file.write(
        'local function storeProps(item, prop, value, mode, tbl)\n'
        '\ttbl[item] = tbl[item] or {}\n'
        '\ttbl[item][prop] = tbl[item][prop] or {}\n'
        '\tif tbl[item][prop][mode] == nil then\n'
        '\t\ttbl[item][prop][mode] = value\n'
        '\tend\n'
        'end\n\n'
    )


def write_lua_function(lua_file, rows, is_tog, function_name, print_label, legacy=False):
    """Escreve uma funcao Lua completa (function ... end) no arquivo ja aberto."""
    lua_file.write(f"function {function_name}()\n")
    lua_file.write(f'\tprint("Running {print_label}...")\n')
    for w in rows:
        snippet = get_weapon_lua_snippet(w, is_tog, is_legacy=legacy)
        print(snippet)
        lua_file.write(snippet)
        lua_file.write("\n\n")
    lua_file.write("\tGBO_ApplyCHTModeItemProps(GBO_gCTHModeItemPropertyTable)\n\n")
    lua_file.write(f'\tprint("{print_label} done")\n')
    lua_file.write("end\n\n")


tog_rows = []
gbo_rows = []
legacy_rows = []
ignored = 0

for row in reader:
    origem = (row.get(ORIGEM_COL) or "").strip()
    if origem == ORIGEM_TOG_PATCHED:
        tog_rows.append(row)
    elif origem == ORIGEM_LEGACY:
        legacy_rows.append(row)
    elif origem == ORIGEM_VANILLA:
        gbo_rows.append(row)
    elif origem == ORIGEM_NON_PATCHED:
        ignored += 1
    else:
        print(f'Aviso: linha "{row.get(ID)}" com #Origem desconhecido/vazio ("{origem}"), ignorada.')
        ignored += 1

print(f"{len(tog_rows)} armas -> TOG patched, {len(legacy_rows)} armas -> Legacy TOG, {len(gbo_rows)} armas -> GBO, {ignored} ignoradas (non-patched ou origem desconhecida).")

with open(LUA_FILE_NAME, "w") as lua_file:
    write_lua_header(lua_file)
    write_lua_function(lua_file, tog_rows, is_tog=True,
                       function_name="RatoTOG_Patch", print_label="RatoTOG_Patch")
    write_lua_function(lua_file, legacy_rows, is_tog=True, legacy=True,
                       function_name="RatoTOG_LegacyPatch", print_label="RatoTOG_LegacyPatch")
    write_lua_function(lua_file, gbo_rows, is_tog=False,
                       function_name="RatoGBO_WepPatch", print_label="PATCH_GBO_weapons")

files.download(LUA_FILE_NAME)


48 armas -> TOG patched, 1 armas -> Legacy TOG, 45 armas -> GBO, 58 ignoradas (non-patched ou origem desconhecida).
	PKM_1.is_tog_patched = true
	PKM_1.is_vanilla_firearm = false
	PKM_1.DisplayName = Untranslated([[PKM]])
	PKM_1.object_class = "RPK74"
	PKM_1.ItemType = "MachineGun"
	PKM_1.Cost = 7200
	PKM_1.Damage = 32
	PKM_1.ObjDamageMod = 100
	PKM_1.CritChance = 0
	PKM_1.CritPerAim = 200
	PKM_1.BurstCritMul = 60
	PKM_1.CritChanceScaled = 10
	storeProps(PKM_1, "AimAccuracy", 31, "aCTH", GBO_gCTHModeItemPropertyTable)
	storeProps(PKM_1, "AimAccuracy", 2, "oldCTH", GBO_gCTHModeItemPropertyTable)
	storeProps(PKM_1, "WeaponRange", 32, "aCTH", GBO_gCTHModeItemPropertyTable)
	storeProps(PKM_1, "WeaponRange", 42, "oldCTH", GBO_gCTHModeItemPropertyTable)
	PKM_1.PBbonus_base = -4
	PKM_1.Handling = -15
	PKM_1.HandlingNotProneMul = 200
	PKM_1.PenetrationClass = 1
	PKM_1.ShootAP = 4000
	PKM_1.ReloadAP = 5000
	PKM_1.APStance = 42
	PKM_1.OverwatchAngle = 583
	PKM_1.MagazineSize = 100
	PKM_1.Cumbers

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>